# Topic 1: Window Operations

> **Phase 6 → Week 11 → Topic 1**
>
> Prerequisites: Week 10 (Watermarking, Stateful aggregations)

---

## Three Types of Windows

```
1. Tumbling Windows (fixed, non-overlapping):
   ├──── 10:00–10:10 ────┤├──── 10:10–10:20 ────┤├──── 10:20–10:30 ────┤
   Each event belongs to exactly ONE window.
   window("ts", "10 minutes")

2. Sliding Windows (fixed size, slide interval):
   ├──── 10:00–10:10 ────┤
      ├──── 10:05–10:15 ────┤
         ├──── 10:10–10:20 ────┤
   Each event can belong to MULTIPLE windows.
   window("ts", "10 minutes", "5 minutes")   ← window_size, slide_interval

3. Session Windows (dynamic, gap-based):
   Events with < gap_duration between them form ONE session.
   If gap > threshold → new session starts.
   session_window("ts", "5 minutes")   ← close session after 5 min of inactivity
   
   User A: click@10:01, click@10:03, click@10:08 → session [10:01–10:08]
            (idle for 8 min)
   User A: click@10:16 → new session [10:16–...]
```

---

## Interview Cheat Sheet

**Q: What is the difference between tumbling and sliding windows?**
> Tumbling windows are fixed-size, non-overlapping intervals. Each event belongs to exactly one window. `window("ts", "10 minutes")`. Sliding windows overlap: `window("ts", "10 minutes", "5 minutes")` creates windows every 5 minutes, each spanning 10 minutes. An event within a 10-minute period appears in two overlapping windows. Sliding is useful for rolling averages (e.g., revenue in the last 10 minutes, computed every 5 minutes).

**Q: What is a session window?**
> A session window groups events that occur within a gap threshold of each other. If no events arrive within the gap duration, the session closes. Sessions are variable-length and user-driven — useful for click-stream analysis, user behavior tracking, and mobile app session analytics. In Spark, `session_window("ts", "5 minutes")` closes a session after 5 minutes of inactivity.

**Q: Do you need a watermark for window aggregations?**
> You need a watermark if you want bounded state (memory safety) or if you want to use `append` mode (which only emits results when the window is closed). Without watermark, window state accumulates forever and you must use `complete` mode. For production, always use watermark with window aggregations.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time, os, shutil
from datetime import datetime, timedelta

spark = SparkSession.builder \
    .appName("Week11 - Window Operations") \
    .master("local[4]") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("Spark", spark.version)

## Part 1: Tumbling Windows

In [ ]:
# Tumbling window: fixed 10-second non-overlapping windows
stream = spark.readStream \
    .format("rate") \
    .option("rowsPerSecond", 10) \
    .load() \
    .withColumn("amount",   (F.col("value") % 100 + 1).cast("double")) \
    .withColumn("category", F.element_at(
        F.array(F.lit("electronics"), F.lit("clothing"), F.lit("food")),
        (F.col("value") % 3 + 1).cast("int")))

# Tumbling window: one result per 10-second window per category
tumbling = stream \
    .withWatermark("timestamp", "10 seconds") \
    .groupBy(
        F.window("timestamp", "10 seconds"),   # tumbling: size only, no slide
        "category"
    ).agg(
        F.count("*").alias("orders"),
        F.round(F.sum("amount"), 2).alias("revenue")
    ) \
    .select(
        F.col("window.start").alias("window_start"),
        F.col("window.end").alias("window_end"),
        "category", "orders", "revenue"
    )

q = tumbling.writeStream \
    .format("memory") \
    .queryName("tumbling_results") \
    .outputMode("update") \
    .trigger(processingTime="5 seconds") \
    .start()

time.sleep(30)
print("Tumbling window results (10-second buckets):")
spark.sql("""
    SELECT date_format(window_start, 'HH:mm:ss') as start,
           date_format(window_end, 'HH:mm:ss') as end,
           category, orders, revenue
    FROM tumbling_results
    ORDER BY window_start, category
""").show(20)

q.stop()

## Part 2: Sliding Windows

In [ ]:
# Sliding window: 20-second window, sliding every 10 seconds
# Each event appears in 2 windows (window_size / slide_interval = 2)

stream2 = spark.readStream \
    .format("rate") \
    .option("rowsPerSecond", 5) \
    .load() \
    .withColumn("amount", (F.col("value") % 100 + 1).cast("double"))

sliding = stream2 \
    .withWatermark("timestamp", "20 seconds") \
    .groupBy(
        F.window("timestamp", "20 seconds", "10 seconds")  # size=20s, slide=10s
    ).agg(
        F.count("*").alias("events"),
        F.round(F.avg("amount"), 2).alias("avg_amount"),
        F.round(F.sum("amount"), 2).alias("total")
    ) \
    .select(
        F.col("window.start").alias("start"),
        F.col("window.end").alias("end"),
        "events", "avg_amount", "total"
    )

q2 = sliding.writeStream \
    .format("memory") \
    .queryName("sliding_results") \
    .outputMode("update") \
    .trigger(processingTime="5 seconds") \
    .start()

time.sleep(40)
print("Sliding window results (20s window, 10s slide):")
print("Note: each 10s period's events appear in TWO windows")
spark.sql("""
    SELECT date_format(start, 'HH:mm:ss') as win_start,
           date_format(end, 'HH:mm:ss') as win_end,
           events, avg_amount, total
    FROM sliding_results
    ORDER BY start
""").show()

q2.stop()

## Part 3: Session Windows (Spark 3.2+)

In [ ]:
# Session window: group events within a gap threshold
# If no events for > gap_duration → close current session, start new one

# Simulate user click events
USER_EVENTS = "/tmp/session_events"
if os.path.exists(USER_EVENTS): shutil.rmtree(USER_EVENTS)
os.makedirs(USER_EVENTS)

now = datetime.now()
def ts(seconds_ago):
    return (now - timedelta(seconds=seconds_ago)).strftime("%Y-%m-%d %H:%M:%S.%f")

# User A: 3 clicks close together (same session) then a gap → new session
# User B: 2 clicks (same session)
events = spark.createDataFrame([
    ("userA", ts(120), "click"),   # session 1 start
    ("userA", ts(115), "click"),   # session 1 (5s later)
    ("userA", ts(110), "click"),   # session 1 (5s later)
    ("userA", ts(30),  "click"),   # session 2 (80s gap → new session)
    ("userA", ts(25),  "click"),   # session 2
    ("userB", ts(100), "click"),   # session 1
    ("userB", ts(95),  "click"),   # session 1
], ["user_id", "event_time", "action"]) \
    .withColumn("event_time", F.to_timestamp("event_time", "yyyy-MM-dd HH:mm:ss.SSSSSS"))

events.coalesce(1).write.json(USER_EVENTS)

event_schema = StructType([
    StructField("user_id",    StringType()),
    StructField("event_time", TimestampType()),
    StructField("action",     StringType()),
])

event_stream = spark.readStream.schema(event_schema).json(USER_EVENTS)

try:
    from pyspark.sql.functions import session_window
    sessions = event_stream \
        .withWatermark("event_time", "2 minutes") \
        .groupBy(
            "user_id",
            session_window("event_time", "60 seconds")  # 60s gap = new session
        ).agg(
            F.count("*").alias("clicks"),
            F.min("event_time").alias("session_start"),
            F.max("event_time").alias("session_end"),
        )

    q3 = sessions.writeStream \
        .format("memory") \
        .queryName("sessions") \
        .outputMode("update") \
        .option("checkpointLocation", "/tmp/ckpt_sessions") \
        .trigger(once=True) \
        .start()

    q3.awaitTermination()
    print("Session window results:")
    spark.sql("""
        SELECT user_id,
               date_format(session_start, 'HH:mm:ss') as start,
               date_format(session_end, 'HH:mm:ss') as end,
               clicks
        FROM sessions ORDER BY user_id, session_start
    """).show()
    print("Expected: userA has 2 sessions (3 clicks + 2 clicks), userB has 1 session (2 clicks)")

except ImportError:
    print("session_window requires Spark 3.2+")
    print("Your Spark version:", spark.version)
    print("Concept: session_window(\"event_time\", \"60 seconds\")")
    print("Groups events where gap between consecutive events < 60 seconds")

## Part 4: Window Reference

In [ ]:
print("""
Window Function Reference:
════════════════════════════════════════════════════════════════

Import:
  from pyspark.sql.functions import window, session_window

Tumbling (non-overlapping):
  F.window("timestamp_col", "10 minutes")
  F.window("timestamp_col", "1 hour")
  F.window("timestamp_col", "1 day")

Sliding (overlapping):
  F.window("timestamp_col", "10 minutes", "5 minutes")  # size, slide
  Rule: slide_interval <= window_size
  Events per window = window_size / slide_interval copies

Session (gap-based, Spark 3.2+):
  session_window("timestamp_col", "5 minutes")  # gap duration
  session_window("timestamp_col", F.lit("5 minutes"))  # dynamic gap

Access window bounds:
  .select(
      F.col("window.start").alias("win_start"),
      F.col("window.end").alias("win_end"),
      ...
  )

Always pair with watermark:
  stream
    .withWatermark("timestamp", "1 hour")   # late data tolerance
    .groupBy(F.window("timestamp", "1 hour"), "key")
    .agg(F.count("*"))

Output mode:
  update: emit partial results as data arrives (low latency)
  append: emit window result ONLY when window is closed (no late data)
════════════════════════════════════════════════════════════════
""")

## Exercises

1. Create a tumbling window aggregation that counts orders and total revenue per **1-minute** window per category. Use `update` mode. Run for 2 minutes and show the results grouped by window.
2. Create a sliding window that computes the 30-second rolling average of `amount`, updated every 10 seconds. How many windows does each event appear in?
3. You have a tumbling 5-minute window with a 2-minute watermark. A window covers 10:00–10:05. When does Spark finalize this window (evict its state) in `update` mode? In `append` mode?
4. What is the memory cost of sliding windows vs tumbling windows? If your window is 1 hour and your slide is 1 minute, how many copies of each event are in state simultaneously?
5. Explain the difference between `F.window()` (stream windowing) and `Window` (batch window functions from Week 4). They have the same name but do completely different things.